# Train a ~30M GPT-style model on your prepped Wikipedia data

**Before running:** Settings -> Accelerator = **GPU T4 x2** (or P100). Internet doesn't need to be on for this notebook.

**Where your data lives:** your `meta.json`/`tokenizer.json`/`train.bin`/`val.bin` dataset should be attached under Input on the right sidebar. The first config cell searches `/kaggle/input` recursively for `meta.json`.

**Checkpointing:** every `CKPT_MINUTES` (default 15) the model, optimizer state, and step count are saved to `/kaggle/working`, which Kaggle keeps for the notebook's outputs. Re-running the notebook resumes automatically from the last checkpoint — this is what makes multi-session training on Kaggle's free weekly GPU quota workable. Save a Version after each session (or periodically) so the checkpoint files in Output aren't lost when the session ends.

This is set to run the real training directly (`TEST_RUN = False`). Before starting a long unattended run, it's still worth watching the first ~50 steps to confirm the loss is actually decreasing and nothing throws an error.

In [ ]:
!pip install -q -U torch --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || true
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import os, sys, math, json, time, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------- FIND YOUR DATA (Kaggle) -------------------------
# Recursively searches every attached dataset under /kaggle/input for meta.json,
# since Kaggle sometimes nests datasets an extra folder deep.
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, (
    "couldn't find meta.json anywhere under /kaggle/input. "
    "Make sure your dataset is attached (right sidebar -> Add Input), then run:\n"
    "  !find /kaggle/input -name meta.json\n"
    "and set DATA_DIR manually to the folder it's in."
)
if len(hits) > 1:
    print("found meta.json in multiple places, using the first one:")
    for h in hits:
        print(" ", h)
DATA_DIR = os.path.dirname(hits[0])
print("using data from:", DATA_DIR)

meta = json.load(open(f"{DATA_DIR}/meta.json"))
print(json.dumps(meta, indent=2))
VOCAB_SIZE = meta["vocab_size"]

## Config
8 layers, 512 width, 8 heads, context 512, tied embeddings -> ~30M params at a 16k vocab.

In [ ]:
# ------------------------- CONFIG -------------------------
TEST_RUN = False          # running straight to the real run, as decided

BLOCK_SIZE = 512          # context length
N_LAYER = 8
N_HEAD = 8
N_EMBD = 512
DROPOUT = 0.0             # you have plenty of data relative to model size; dropout not needed

BATCH_SIZE = 16           # lowered for OOM safety; the batch-size probe cell will tell you if you can safely raise it
GRAD_ACCUM = 8            # doubled to roughly keep the effective batch size the same as before

MAX_LR = 1e-3
MIN_LR = MAX_LR * 0.1
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

CKPT_MINUTES = 15
EVAL_EVERY_STEPS = 200
EVAL_ITERS = 50           # batches averaged for val loss
SAMPLE_EVERY_STEPS = 500

if TEST_RUN:
    MAX_STEPS = 300
    WARMUP_STEPS = 20
    RUN_NAME = "test"
else:
    # ---- SET THIS from your real token budget and measured tokens/sec (see the speed-test cell) ----
    # A reasonable first real run: ~150-300M tokens (not the full 1B) so it finishes in hours, not weeks,
    # on a free Kaggle GPU. You can always resume and train further later if you want more.
    # tokens_per_step = BATCH_SIZE * GRAD_ACCUM * n_gpu * BLOCK_SIZE (printed below).
    # MAX_STEPS = target_tokens // tokens_per_step
    MAX_STEPS = 6000
    WARMUP_STEPS = 150
    RUN_NAME = "run1"

SAVE_DIR = f"/kaggle/working/ckpt_{RUN_NAME}"   # /kaggle/input is read-only; checkpoints must go in /kaggle/working
os.makedirs(SAVE_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpu = torch.cuda.device_count()
print(f"device={device}, n_gpu={n_gpu}, save_dir={SAVE_DIR}")
print(f"effective batch (tokens/step) = {BATCH_SIZE * GRAD_ACCUM * max(1,n_gpu) * BLOCK_SIZE:,}")

## Data loading
Memmap so `train.bin`/`val.bin` are never fully loaded into RAM.

In [ ]:
train_data = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap(f"{DATA_DIR}/val.bin", dtype=np.uint16, mode="r")
print(f"train tokens: {len(train_data):,} | val tokens: {len(val_data):,}")

def get_batch(split, batch_size):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+BLOCK_SIZE].astype(np.int64)) for i in ix])
    if device == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train", 4)
print("batch shapes:", xb.shape, yb.shape, "| max id:", int(xb.max()), "(must be <", VOCAB_SIZE, ")")

## Model
Minimal GPT (nanoGPT-style): causal self-attention via `F.scaled_dot_product_attention` (uses Flash Attention when available), tied input/output embeddings.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.n_embd, self.dropout = n_head, n_embd, dropout
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_drop = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                            dropout_p=self.attn_drop if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight   # weight tying: saves ~vocab_size*n_embd params
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

def count_params(model):
    return sum(p.numel() for p in model.parameters())

model = GPT(VOCAB_SIZE, BLOCK_SIZE, N_LAYER, N_HEAD, N_EMBD, DROPOUT).to(device)
n_params = count_params(model)
print(f"parameters: {n_params:,} ({n_params/1e6:.1f}M)")

if n_gpu > 1:
    model = nn.DataParallel(model)   # simple multi-GPU for Kaggle's 2x T4; each step splits the batch across GPUs
raw_model = model.module if n_gpu > 1 else model

## Optimizer, LR schedule, checkpoint/resume

In [ ]:
decay, no_decay = [], []
for n, p in raw_model.named_parameters():
    (no_decay if p.dim() < 2 else decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
    lr=MAX_LR, betas=(0.9, 0.95),
)

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS
    if step >= MAX_STEPS:
        return MIN_LR
    prog = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1 + math.cos(math.pi * prog))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

CKPT_PATH = f"{SAVE_DIR}/latest.pt"

def save_ckpt(step, best_val):
    tmp = CKPT_PATH + ".tmp"
    torch.save({
        "step": step, "best_val": best_val,
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "config": dict(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_layer=N_LAYER,
                        n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT),
    }, tmp)
    os.replace(tmp, CKPT_PATH)   # atomic: never leaves a half-written checkpoint

start_step, best_val = 0, float("inf")
if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=device)
    raw_model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_step, best_val = ck["step"], ck["best_val"]
    print(f"RESUMED from step {start_step} (best val loss so far: {best_val:.4f})")
else:
    print("no checkpoint found, starting fresh")

## Find a safe batch size (run this once, before the real run)
Finds the largest `BATCH_SIZE` that fits in GPU memory by trying increasing sizes, so you don't discover an
out-of-memory error hours into the real run. Uses the current model, so run it after the model cell above.
It restores the model/optimizer to a fresh state afterward (the probing steps don't count as real training).

In [ ]:
def try_batch_size(bs):
    try:
        xb, yb = get_batch("train", bs)
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(xb, yb)
        loss.mean().backward()
        optimizer.step()
        if device == "cuda":
            torch.cuda.synchronize()
        return True
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            return False
        raise
    finally:
        optimizer.zero_grad(set_to_none=True)
        if device == "cuda":
            torch.cuda.empty_cache()

if device == "cuda":
    candidates = [8, 16, 24, 32, 48, 64, 96, 128]
    safe = None
    for bs in candidates:
        ok = try_batch_size(bs)
        print(f"BATCH_SIZE={bs}: {'OK' if ok else 'OUT OF MEMORY'}")
        if not ok:
            break
        safe = bs
    assert safe is not None, "even the smallest batch size (8) OOM'd - reduce N_EMBD/N_LAYER/BLOCK_SIZE"
    recommended = max(8, int(safe * 0.8))   # leave headroom: activations grow a bit once eval/sampling also run
    print(f"\nlargest working batch size: {safe} | recommended BATCH_SIZE (with headroom): {recommended}")
    print("If this differs from the BATCH_SIZE set in the config cell above, update it there and re-run from that cell.")

    # reset model/optimizer so the probing steps above don't count as real training
    model = GPT(VOCAB_SIZE, BLOCK_SIZE, N_LAYER, N_HEAD, N_EMBD, DROPOUT).to(device)
    if n_gpu > 1:
        model = nn.DataParallel(model)
    raw_model = model.module if n_gpu > 1 else model
    decay, no_decay = [], []
    for n, p in raw_model.named_parameters():
        (no_decay if p.dim() < 2 else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
        lr=MAX_LR, betas=(0.9, 0.95),
    )
    print("model and optimizer reset to a fresh state (probing steps discarded)")
else:
    print("no GPU detected - skipping batch size probe (CPU runs are for testing only)")

## Quick speed test
Run this once on the real data (`TEST_RUN` either way) to get real tokens/sec, then use it to size `MAX_STEPS` for your target token budget or time window.

In [ ]:
@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch("val", BATCH_SIZE)
        _, loss = model(xb, yb)
        losses.append(loss.mean().item())
    model.train()
    return sum(losses) / len(losses)

model.train()
t0 = time.time()
n_tok = 0
for _ in range(20):
    for _ in range(GRAD_ACCUM):
        xb, yb = get_batch("train", BATCH_SIZE)
        _, loss = model(xb, yb)
        (loss.mean() / GRAD_ACCUM).backward()
        n_tok += xb.numel()
    optimizer.step(); optimizer.zero_grad(set_to_none=True)
dt = time.time() - t0
tok_per_sec = n_tok / dt
print(f"~{tok_per_sec:,.0f} tokens/sec  ({dt/20*1000:.0f} ms/step)")
if not TEST_RUN:
    print(f"time for MAX_STEPS={MAX_STEPS}: "
          f"{MAX_STEPS * BATCH_SIZE*GRAD_ACCUM*max(1,n_gpu)*BLOCK_SIZE / tok_per_sec / 3600:.1f} hours")
    print("If that's too long/short, adjust MAX_STEPS above and re-run from the config cell "
          "(this restarts the run - only do this before real training starts).")

## Train
This is the main loop. It checkpoints every `CKPT_MINUTES`, evaluates val loss periodically, and prints a text sample so you can watch it go from gibberish to real words. Safe to stop and re-run the whole notebook at any time - it resumes from the last checkpoint.

In [ ]:
def decode_sample(ids):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(f"{DATA_DIR}/tokenizer.json")
    return tok.decode(ids)

def generate_sample():
    ctx = torch.zeros((1, 1), dtype=torch.long, device=device)  # start from token 0
    out = raw_model.generate(ctx, max_new_tokens=100, temperature=0.8, top_k=50)[0].tolist()
    print("--- sample ---")
    print(decode_sample(out))
    print("--------------")

last_ckpt_time = time.time()
t_start = time.time()
step = start_step
while step < MAX_STEPS:
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    try:
        for _ in range(GRAD_ACCUM):
            xb, yb = get_batch("train", BATCH_SIZE)
            _, loss = model(xb, yb)
            (loss.mean() / GRAD_ACCUM).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            save_ckpt(step, best_val)  # save what we have before raising
            raise RuntimeError(
                f"OOM at step {step} with BATCH_SIZE={BATCH_SIZE}. Checkpoint was saved at step {step}. "
                f"Lower BATCH_SIZE in the config cell (or re-run the batch-size probe cell), "
                f"then re-run from the config cell - it will resume from step {step}."
            ) from e
        raise
    step += 1

    if step % 20 == 0:
        elapsed = time.time() - t_start
        print(f"step {step}/{MAX_STEPS} | loss {loss.mean().item():.4f} | lr {lr:.2e} | {elapsed/60:.1f} min elapsed")

    if step % EVAL_EVERY_STEPS == 0 or step == MAX_STEPS:
        vl = estimate_val_loss()
        print(f"  [eval] step {step} | val loss {vl:.4f}")
        best_val = min(best_val, vl)

    if step % SAMPLE_EVERY_STEPS == 0:
        generate_sample()

    if time.time() - last_ckpt_time >= CKPT_MINUTES * 60 or step == MAX_STEPS:
        save_ckpt(step, best_val)
        last_ckpt_time = time.time()
        print(f"  [checkpoint] saved at step {step}")

print("done. final val loss (best seen):", best_val)

## After training
- Kaggle sessions cap out at ~9-12 hrs and the free weekly GPU quota is limited — just re-run the whole notebook in your next session, it resumes from `ckpt_run1/latest.pt` automatically.
- Click **Save Version** periodically (not just at the end) so the checkpoint files in `/kaggle/working` survive between sessions.
- When `best_val` stops improving for a while, you're done. Download `ckpt_run1/latest.pt` and `tokenizer.json` from the Output panel — that's your trained model.